<a href="https://colab.research.google.com/github/itsbhh/IoT-BASED-AIR-QUALITY-MONITORING-SYSTEM/blob/main/IoT%20BASED%20AIR%20QUALITY%20MONITORING%20SYSTEM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ==================================================
# IoT BASED AIR QUALITY MONITORING SYSTEM
# ESP32 + MQ135 + PMS5003
# GOOGLE COLAB SIMULATION
# ==================================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

# 1. GENERATE SENSOR DATA

np.random.seed(42)

n = 30

data = pd.DataFrame({
    "Time": range(1, n + 1),
    "MQ135": np.random.randint(500, 3000, n),
    "PM1.0": np.random.randint(5, 80, n),
    "PM2.5": np.random.randint(10, 150, n),
    "PM10": np.random.randint(20, 250, n)
})

data["PM2.5"] = np.maximum(data["PM2.5"], data["PM1.0"])
data["PM10"] = np.maximum(data["PM10"], data["PM2.5"])


# 2. AIR QUALITY CLASSIFICATION

def air_quality(pm25):

    if pm25 <= 15:
        return "GOOD"

    elif pm25 <= 35:
        return "MODERATE"

    else:
        return "POOR"


data["Status"] = data["PM2.5"].apply(air_quality)


# 3. DASHBOARD FUNCTION

def show_dashboard():

    latest = data.iloc[-1]

    status = latest["Status"]

    colors = {
        "GOOD": "#16a34a",
        "MODERATE": "#e69b18",
        "POOR": "#dc2626"
    }

    color = colors[status]

    display(HTML(f"""
    <div style="background:#172554;color:white;
    padding:18px;border-radius:12px;text-align:center">
    <h2>IoT AIR QUALITY MONITORING SYSTEM</h2>
    <p>ESP32 | MQ-135 | PMS5003</p>
    <h1 style="color:{color}">{status}</h1>
    <h4>Current Air Quality Status</h4>
    </div>
    """))

    # 4. DISPLAY FOUR GAUGES

    fig = make_subplots(
        rows=2,
        cols=2,
        specs=[
            [{"type": "indicator"}, {"type": "indicator"}],
            [{"type": "indicator"}, {"type": "indicator"}]
        ],
        vertical_spacing=0.18,
        horizontal_spacing=0.12
    )

    gauges = [
        ("MQ-135", latest["MQ135"], 4095, "Raw ADC"),
        ("PM1.0", latest["PM1.0"], 100, "µg/m³"),
        ("PM2.5", latest["PM2.5"], 250, "µg/m³"),
        ("PM10", latest["PM10"], 400, "µg/m³")
    ]

    for i, (name, value, maximum, unit) in enumerate(gauges):

        row = i // 2 + 1
        col = i % 2 + 1

        fig.add_trace(
            go.Indicator(
                mode="gauge+number",
                value=float(value),
                title={
                    "text": f"{name}<br>{unit}",
                    "font": {"size": 16}
                },
                number={"font": {"size": 30}},
                gauge={
                    "axis": {
                        "range": [0, maximum],
                        "tickfont": {"size": 10}
                    },
                    "bar": {"color": color},
                    "borderwidth": 1
                }
            ),
            row=row,
            col=col
        )

    fig.update_layout(
        title={
            "text": "SENSOR READINGS",
            "x": 0.5
        },
        height=650,
        margin=dict(l=50, r=50, t=80, b=40),
        paper_bgcolor="white",
        font={"color": "#24344d"}
    )

    fig.show()


    # 5. POLLUTION HISTORY GRAPH

    graph = go.Figure()

    for col in ["PM1.0", "PM2.5", "PM10"]:

        graph.add_trace(
            go.Scatter(
                x=data["Time"],
                y=data[col],
                mode="lines+markers",
                name=col
            )
        )

    graph.update_layout(
        title="AIR POLLUTION MONITORING HISTORY",
        xaxis_title="Time / Reading Number",
        yaxis_title="Concentration (µg/m³)",
        height=400,
        template="plotly_white",
        margin=dict(l=50, r=30, t=60, b=50)
    )

    graph.show()


    # 6. SENSOR DATA TABLE

    print("\n========== SENSOR DATA ==========")

    display(data.tail(10).reset_index(drop=True))


    # 7. SENSOR ALERT

    print("\n========== AIR QUALITY ALERT ==========")

    if status == "POOR":
        print("WARNING: HIGH AIR POLLUTION DETECTED!")

    elif status == "MODERATE":
        print("NOTICE: MODERATE AIR POLLUTION.")

    else:
        print("AIR QUALITY IS GOOD.")


# 8. DISPLAY INITIAL DASHBOARD

show_dashboard()



# ==================================================
# 9. ADD YOUR OWN SENSOR DATA OPTION
# ==================================================

print("\n======================================")
print("       ADD YOUR OWN SENSOR DATA")
print("======================================")

print("""
SENSOR INPUT GUIDE
--------------------------------------
1. MQ-135 : 0 to 4095 (Example: 2300)
2. PM1.0  : 0 to 100 µg/m³ (Example: 25)
3. PM2.5  : 0 to 250 µg/m³ (Example: 65)
4. PM10   : 0 to 400 µg/m³ (Example: 150)

NOTE:
- MQ-135 value is a raw ADC reading, NOT ppm.
- PM values are particulate concentrations.
- Ensure PM1.0 <= PM2.5 <= PM10.
--------------------------------------
""")

choice = input(
    "\nDo you want to add your own data and check? (yes/no): "
).strip().lower()

if choice == "yes":

    print("\nENTER YOUR SENSOR READINGS")

    mq = float(input(
        "MQ-135 (0-4095, e.g. 2300): "
    ))

    pm1 = float(input(
        "PM1.0 (0-100, e.g. 25): "
    ))

    pm25 = float(input(
        "PM2.5 (0-250, e.g. 65): "
    ))

    pm10 = float(input(
        "PM10 (0-400, e.g. 150): "
    ))

    if not (
        0 <= mq <= 4095 and
        0 <= pm1 <= 100 and
        0 <= pm25 <= 250 and
        0 <= pm10 <= 400
    ):

        print("\nInvalid input! Please follow the given ranges.")

    elif not (pm1 <= pm25 <= pm10):

        print("\nInvalid readings!")
        print("PM1.0 must be <= PM2.5 <= PM10.")

    else:

        new_status = air_quality(pm25)

        new_row = {
            "Time": len(data) + 1,
            "MQ135": mq,
            "PM1.0": pm1,
            "PM2.5": pm25,
            "PM10": pm10,
            "Status": new_status
        }

        data.loc[len(data)] = new_row

        print("\nYOUR DATA HAS BEEN ADDED SUCCESSFULLY!")

        print("\nUPDATED AIR QUALITY RESULT:")

        show_dashboard()

elif choice == "no":

    print("\nShowing simulated sensor results only.")

else:

    print("\nPlease enter yes or no and run again.")




# ==================================================
# END OF PROJECT SIMULATION
# ==================================================



========== SENSOR DATA ==========


,Time,MQ135,PM1.0,PM2.5,PM10,Status
0,21,1684,64,98,123,POOR
1,22,959,75,80,233,POOR
2,23,521,48,48,238,POOR
3,24,2800,12,97,97,POOR
4,25,1247,51,138,212,POOR
5,26,974,39,145,246,POOR
6,27,1582,40,72,120,POOR
7,28,2547,54,148,194,POOR
8,29,1475,8,90,225,POOR
9,30,2306,6,145,150,POOR



========== AIR QUALITY ALERT ==========

       ADD YOUR OWN SENSOR DATA

SENSOR INPUT GUIDE
--------------------------------------
1. MQ-135 : 0 to 4095 (Example: 2300)
2. PM1.0  : 0 to 100 µg/m³ (Example: 25)
3. PM2.5  : 0 to 250 µg/m³ (Example: 65)
4. PM10   : 0 to 400 µg/m³ (Example: 150)

NOTE:
- MQ-135 value is a raw ADC reading, NOT ppm.
- PM values are particulate concentrations.
- Ensure PM1.0 <= PM2.5 <= PM10.
--------------------------------------


Do you want to add your own data and check? (yes/no): YES

ENTER YOUR SENSOR READINGS
MQ-135 (0-4095, e.g. 2300): 2300
PM1.0 (0-100, e.g. 25): 25
PM2.5 (0-250, e.g. 65): 65
PM10 (0-400, e.g. 150): 140

YOUR DATA HAS BEEN ADDED SUCCESSFULLY!

UPDATED AIR QUALITY RESULT:



========== SENSOR DATA ==========


,Time,MQ135,PM1.0,PM2.5,PM10,Status
0,22,959.0,75.0,80.0,233.0,POOR
1,23,521.0,48.0,48.0,238.0,POOR
2,24,2800.0,12.0,97.0,97.0,POOR
3,25,1247.0,51.0,138.0,212.0,POOR
4,26,974.0,39.0,145.0,246.0,POOR
5,27,1582.0,40.0,72.0,120.0,POOR
6,28,2547.0,54.0,148.0,194.0,POOR
7,29,1475.0,8.0,90.0,225.0,POOR
8,30,2306.0,6.0,145.0,150.0,POOR
9,31,2300.0,25.0,65.0,140.0,POOR



========== AIR QUALITY ALERT ==========
